# Wave Equation and C Code Generation

Turn the scalar-wave right-hand sides into NRPy symbolic expressions and complete
C assignments.

Navigation: [Index](../index.ipynb) |
Previous: [Finite-Difference Playground][fd-playground] |
Next: [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)

[fd-playground]: ../2-numerical_methods/finite_difference_playground.ipynb

## Learning Goals

- Write the wave equation as a first-order-in-time system.
- Check that NRPy's symbolic right-hand sides match a hand-written formula.
- Generate C assignments for both evolution and exact initial data.

## Words for This Notebook

- **First-order system:** a rewrite where each equation contains only first time
  derivatives.
- **Right-hand side:** the expression after the equals sign in a time-change equation.
- **Exact solution:** a known solution used to test numerical code.
- **Residual:** zero means two formulas match after simplification.
- **Laplacian:** the sum of second spatial derivatives of a field.
- **BHaH:** the NRPy code-writing infrastructure selected in the setup cell.

Use the code cells actively: first predict what should happen, then run the cell,
then explain the output in plain language.

## Symbolic Wave Right-Hand Sides
The first-order scalar wave system uses

$$
\partial_t u = v, \qquad
\partial_t v = c^2(\partial_x^2 u + \partial_y^2 u + \partial_z^2 u).
$$

The next cells build these two right-hand sides with NRPy and compare them
against the hand-written formula.

## Import SymPy for Residual Checks

These imports expose the NRPy and Python tools used in the next steps.

In [1]:
import sympy as sp

## Import Wave Equation and Codegen Tools

These imports expose the NRPy modules used below.

In [2]:
import nrpy.c_codegen as ccg
import nrpy.grid as grid
import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData import (
    WaveEquation_solution_Cartesian,
)

## Step 1: Build the Cartesian Wave Right-Hand Sides

The printed right-hand sides are the symbolic expressions later passed to C code
generation.

In [3]:
for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    grid.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")
rhs = WaveEquation_RHSs()
print("uu_rhs =", rhs.uu_rhs)
print("vv_rhs =", rhs.vv_rhs)

uu_rhs = vv
vv_rhs = uu_dDD00*wavespeed**2 + uu_dDD11*wavespeed**2 + uu_dDD22*wavespeed**2


## Validation Check: Compare Against the Hand Formula

Before running the next cell, write the two hand formulas in words. A zero
residual is the check that the symbolic NRPy expressions represent the same
first-order wave system.

In [4]:
wavespeed = sp.Symbol("wavespeed", real=True)
vv = sp.Symbol("vv", real=True)
uu_dDD = ixp.declarerank2("uu_dDD", symmetry="sym01")
hand_uu_rhs = vv
hand_vv_rhs = wavespeed**2 * sum(uu_dDD[i][i] for i in range(3))
print("uu residual:", sp.simplify(rhs.uu_rhs - hand_uu_rhs))
print("vv residual:", sp.simplify(rhs.vv_rhs - hand_vv_rhs))
if (
    sp.simplify(rhs.uu_rhs - hand_uu_rhs) != 0
    or sp.simplify(rhs.vv_rhs - hand_vv_rhs) != 0
):
    raise RuntimeError(
        "Expected the NRPy right-hand sides to match the hand-written right-hand sides."
    )

uu residual: 0
vv residual: 0


## Where These Assignments Go

The next two code-generation cells show the small C assignment blocks that later
appear inside complete generated projects.

| Assignment block | Later project role | What to inspect |
| --- | --- | --- |
| `uu_rhs`, `vv_rhs` | right-hand-side update in `rhs_eval.c` | fields read and fields updated |
| `uu_exact`, `vv_exact` | exact data and diagnostics | trusted analytic values |

## Step 2: Emit Right-Hand-Side Assignments

These assignments are the compact update body for the evolution equations. In a
full project, finite-difference code wraps this body in a grid loop and writes
the result into right-hand-side grid fields.

In [5]:
rhs_assignments = ccg.c_codegen(
    [rhs.uu_rhs, rhs.vv_rhs],
    ["uu_rhs", "vv_rhs"],
    include_braces=False,
    verbose=False,
)
print("complete right-hand-side assignments:")
print(rhs_assignments)

complete right-hand-side assignments:
const REAL tmp0 = ((wavespeed)*(wavespeed));
uu_rhs = vv;
vv_rhs = tmp0*uu_dDD00 + tmp0*uu_dDD11 + tmp0*uu_dDD22;



## Step 3: Emit Exact-Solution Assignments

These assignments evaluate an analytic wave. Complete projects use the same
idea to set initial data and to compute diagnostic errors against a trusted
solution.

In [6]:
plane_wave = WaveEquation_solution_Cartesian(WaveType="PlaneWave")
exact_solution_assignments = ccg.c_codegen(
    [plane_wave.uu_exactsoln, plane_wave.vv_exactsoln],
    ["uu_exact", "vv_exact"],
    include_braces=False,
    verbose=False,
)
print("complete exact-solution assignments:")
print(exact_solution_assignments)

complete exact-solution assignments:
const REAL tmp0 = time*wavespeed - (kk0*xCart0 + kk1*xCart1 + kk2*xCart2)/sqrt(((kk0)*(kk0)) + ((kk1)*(kk1)) + ((kk2)*(kk2)));
uu_exact = 2 - sin(tmp0);
vv_exact = -wavespeed*cos(tmp0);



The residuals confirm that the generated symbolic right-hand sides match the
hand-written wave system. The C assignments are the compact update body used
by complete projects.

## Learning Check

Explain how the symbolic residual, the right-hand-side assignments, and the
exact-solution assignments support the later generated-project workflow.

## Continue to Cartesian Project Generation
- [C Code Generation](../1-intro/c_codegen.ipynb)
- [Finite Differences](../1-intro/finite_difference.ipynb)
- [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)